In [27]:
// ==== Setup ====

import java.util.*;

// wall-clock time of r, in milliseconds
long millis(Runnable r) {
    System.gc();            // start from a clean heap: less noise
    long start = System.nanoTime();
    r.run();
    return (System.nanoTime() - start) / 1_000_000;
}

// test helpers
void check(boolean ok, String what) {
    System.out.println((ok ? "ok    " : "FAIL  ") + what);
}

# Lookup by key

## Map ADT

- The record of a student, given the NetID.
- The capital of a country, given the country.
- The number of occurrences of a word, given the word.

A collection of **(key, value)** pairs.
Keys are **unique**: each key is mapped to at most one value.
Also called dictionary, associative array, symbol table.

| operation | |
|---|---|
| `put(k, v)` | map `k` to `v`; if `k` is already there, replace its value |
| `get(k)` | the value of `k` |
| `remove(k)` | remove `k` and its value |
| `containsKey(k)` | is `k` in the map? |
| `isEmpty()`, `size()` | |

In [ ]:
interface MyMap<K, V> {
    // return the previous value of key, or null if key was not in the map
    V put(K key, V value);
    V remove(K key);

    // null if key is not in the map
    V get(K key);
    boolean containsKey(K key);

    boolean isEmpty();
    int size();
}

## Implementations so far

Pairs stored in a list, by key:

| structure | `get` | `put` | `remove` |
|---|---|---|---|
| unsorted array-based list | `O(N)` | `O(N)` | `O(N)` |
| sorted array-based list | `O(log N)` | `O(N)` | `O(N)` |
| unsorted linked list | `O(N)` | `O(N)` | `O(N)` |
| sorted linked list | `O(N)` | `O(N)` | `O(N)` |

- `put` on an unsorted list: why not `O(1)`?
- Sorted linked list: why not `O(log N)` for `get`?
- Goal: all three in `O(1)`.
- Price: extra memory, and `O(1)` only **on average**.

## `java.util.Map<K, V>`

Same operations, implemented by `HashMap<K, V>` (and others).

In [29]:
Map<String, String> capital = new HashMap<>();
capital.put("Italy", "Rome");
capital.put("Japan", "Tokyo");
capital.put("France", "Paris");
capital.put("Portugal", "Lisbon");

System.out.println(capital.put("Italy", "Florence") + " " + capital.get("Italy"));
System.out.println(capital.get("Spain") + " " + capital.containsKey("Spain"));
System.out.println(capital.remove("Japan") + " " + capital.size());

for (String country : capital.keySet())
    System.out.println(country + " -> " + capital.get(country));

Rome Florence
null false
Tokyo 3
Italy -> Florence
France -> Paris
Portugal -> Lisbon


- `get(k)` returns `null`: is `k` absent, or mapped to `null`?
- The order of `keySet()`: insertion order? Alphabetical?

# From keys to indices

## Direct addressing

- Keys are integers in `0..M-1`: an array of size `M`, the value of `k` in `data[k]`.
- `put`, `get`, `remove`: `O(1)`, always.
- Example: scores of the students, by seat number `0..99`.

The key may be:
- too large: 9-digit student IDs, for 300 students
- negative
- not an integer: `String`, `Point`, ...

## Hash function

```
key  ──hash──▶  int  ──mod M──▶  index in 0..M-1
```

- **Collision**: two different keys, same index.
- More possible keys than cells: collisions are unavoidable.

A good hash function is:
- **fast** to compute, and O(1)
- **deterministic**: the same key gives the same result (in the same program run)
- **uniform**: spreads the keys evenly over the cells

An injective hash function over a set of keys S (known in advance) is called a **perfect** hash function, as it gives no collisions.

## `equals` and `hashCode`

- A hash table compares keys with `equals`.
- **Contract**: if `a.equals(b)`, then `a.hashCode() == b.hashCode()`.
- The converse is not required: equal hash codes, different objects is a collision.
- **Whenever you override `equals`, you must also override `hashCode`.**

## `hashCode()` in Java

- `int hashCode()` is defined in `Object`: every object has one.

### `Object.hashCode()`
How can we hash a generic object?
- Can we use its fields? No, `Object` has no idea of what fields its subclasses implement;
- Can we just hash its memory address? No, because the GC is allowed to move things in heap memory to keep it efficient, so addresses can change;
- So a solution used by some JVM is to use a random number, generated at the first call and stored in the header of the object;
- In general, the actual implementation is **JVM-dependent** so you should not rely on any particular one.

So for objects we have an **identity** hash code, it does not depend on the fields of the object and holds the same value for the whole life of the object.
`System.identityHashCode(o)` gives the `Object` version, even if the class overrides `hashCode`.

In [34]:
Object a = new Object();
Object b = new Object();
System.out.println(a.hashCode() + " " + a.hashCode() + " " + b.hashCode());

String s1 = "hello";
String s2 = new String("hello");      // equal to s1, a different object
System.out.println(s1.hashCode() + " " + s2.hashCode());
System.out.println(System.identityHashCode(s1) + " " + System.identityHashCode(s2));

1384148699 1384148699 328020023
99162322 99162322
1677257418 970765321


- Different values at every run of the program: a problem for a hash table?
- `Object.hashCode()` is random and well spread: why do `String` and `Integer` override it?

### Overriding `equals` only

- `Object`: `equals` is `==`, `hashCode` is the identity hash code: the contract holds.
- Override `equals` only: two equal objects, two (probably) different cells.

In [ ]:
class Point {
    final int x, y;

    Point(int x, int y) { this.x = x; this.y = y; }

    @Override
    public boolean equals(Object o) {
        if (!(o instanceof Point)) return false;
        Point p = (Point) o;
        return x == p.x && y == p.y;
    }
    // no hashCode: inherited from Object
}

class HPoint {
    final int x, y;

    HPoint(int x, int y) { this.x = x; this.y = y; }

    @Override
    public boolean equals(Object o) {
        if (!(o instanceof HPoint)) return false;
        HPoint p = (HPoint) o;
        return x == p.x && y == p.y;
    }

    @Override
    public int hashCode() { return 31 * x + y; }     // or Objects.hash(x, y)
}

In [ ]:
Set<Point> points = new HashSet<>();
points.add(new Point(1, 2));
System.out.println(new Point(1, 2).equals(new Point(1, 2)) + " " + points.contains(new Point(1, 2)));

Set<HPoint> hpoints = new HashSet<>();
hpoints.add(new HPoint(1, 2));
System.out.println(new HPoint(1, 2).equals(new HPoint(1, 2)) + " " + hpoints.contains(new HPoint(1, 2)));

- `public int hashCode() { return 42; }`: does it respect the contract? Is it a good idea?
- A `HPoint` is used as a key, then its `x` changes. What happens to `get`?

### `Integer` and `String`

- `Integer`: the value itself.
- `String`, length `n`: `s[0]·31^(n-1) + s[1]·31^(n-2) + ... + s[n-1]`, in `int` arithmetic (overflow wraps around).

In [35]:
// same formula as String.hashCode(), computed with Horner's rule
int stringHash(String s) {
    int h = 0;
    for (int i = 0; i < s.length(); i++)
        h = 31 * h + s.charAt(i);
    return h;
}

for (String s : List.of("", "a", "ab", "ba", "hello", "hash table", "data structures"))
    System.out.printf("%-17s %12d %12d%n", "\"" + s + "\"", stringHash(s), s.hashCode());

System.out.println(Integer.valueOf(-7).hashCode() + " " + "Aa".hashCode() + " " + "BB".hashCode());

""                           0            0
"a"                         97           97
"ab"                      3105         3105
"ba"                      3135         3135
"hello"               99162322     99162322
"hash table"        -794915268   -794915268
"data structures"   1410096022   1410096022
-7 2112 2112


- Why not just the sum of the characters?
- `"Aa"` and `"BB"` collide. Another pair of strings that collide?
- Cost of `hashCode()` for a string of length `n`? (`String` caches it after the first call.)

## From hash code to index

- A hash code can be **negative**: `h % M` is then negative.
- `Math.floorMod(h, M)`: always in `0..M-1`.
- `Math.abs(h) % M`?

In [ ]:
int M = 10;
for (String s : List.of("hello", "data structures", "polygenelubricants")) {
    int h = s.hashCode();
    System.out.printf("%-20s %12d %4d %4d %4d%n", s, h, h % M, Math.floorMod(h, M), Math.abs(h) % M);
}

- `Math.abs(Integer.MIN_VALUE)`: why?